In [ ]:
import numpy as np

embedding: np.ndarray = np.load(
    "/home/pasquale/PycharmProjects/Glomeruli-FP03-2026/data/glomeruli/embeddings/dinov2_base_patch_embeddings.npy")

print(embedding.shape)

*Sanity checks*

In [ ]:
from metrics.embedding_evaluation import find_exact_duplicate, evaluate_embedding_norms

duplicates = find_exact_duplicate(embedding)
if (duplicates != None):
    print("Found duplicate embeddings")

result = evaluate_embedding_norms(embedding)

#print(result["summary"])
print("Near zero:", result["near_zero_indices"])
print("Low outliers:", result["low_outlier_indices"])
print("High outliers:", result["high_outlier_indices"])

import matplotlib.pyplot as plt

norms = result["norms"]

plt.hist(norms[np.isfinite(norms)], bins=50)
plt.xlabel("Embedding norm before normalization")
plt.ylabel("Count")
plt.title("Distribution of embedding norms")
plt.show()

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(embedding)

# PCA fit
pca_full = PCA()
pca_full.fit(X_train_scaled)

explained = pca_full.explained_variance_ratio_

pc1_variance_ratio = explained[0]
top5_variance_ratio = explained[:5].sum()
top10_variance_ratio = explained[:10].sum()

# Cumulative variance
cum_var = np.cumsum(pca_full.explained_variance_ratio_)
n_components = np.arange(1, len(cum_var) + 1)

plt.figure(figsize=(8, 4))
plt.plot(n_components, cum_var, linewidth=2)
plt.xlabel("Number of PCA components")
plt.ylabel("Cumulative explained variance")
plt.title("PCA cumulative explained variance")
plt.grid(True)

# Thresholds with distinct colors
thresholds = [0.90, 0.95, 0.98, 0.99]
colors = ["tab:green", "tab:blue", "tab:orange", "tab:red"]

for thr, color in zip(thresholds, colors):
    k = np.argmax(cum_var >= thr) + 1

    # horizontal line
    plt.axhline(y=thr, linestyle='--', color=color, alpha=0.7)

    # vertical line
    plt.axvline(x=k, linestyle='--', color=color, alpha=0.7)

    # highlight point
    plt.scatter(k, cum_var[k - 1], color=color, s=80, zorder=5)

    # annotation
    plt.annotate(
        f"{int(thr * 100)}% → {k}",
        xy=(k, cum_var[k - 1]),
        xytext=(k + len(cum_var) * 0.02, thr - 0.05),
        arrowprops=dict(arrowstyle="->", color=color),
        fontsize=10,
        color=color
    )

plt.tight_layout()
plt.show()

print("Total components:", embedding.shape[1])

print("\nVariance concentration:")
print(f"PC1 variance ratio: {pc1_variance_ratio:.4f} ({pc1_variance_ratio * 100:.2f}%)")
print(f"Top 5 variance ratio: {top5_variance_ratio:.4f} ({top5_variance_ratio * 100:.2f}%)")
print(f"Top 10 variance ratio: {top10_variance_ratio:.4f} ({top10_variance_ratio * 100:.2f}%)")
# Print values
print("\nComponents needed:")
for thr in thresholds:
    k = np.argmax(cum_var >= thr) + 1
    print(f"{int(thr * 100)}% variance -> {k} components")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

x_scaled = StandardScaler().fit_transform(embedding)

pca_2d = PCA(n_components=2)
coords = pca_2d.fit_transform(x_scaled)

print("PC1 explained variance:", pca_2d.explained_variance_ratio_[0])
print("PC2 explained variance:", pca_2d.explained_variance_ratio_[1])
print("PC1 + PC2:", pca_2d.explained_variance_ratio_.sum())

plt.figure(figsize=(7, 6))
plt.scatter(coords[:, 0], coords[:, 1], s=15, alpha=0.7)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA 2D projection")
plt.grid(alpha=0.3)
plt.show()

Distance distribution

In [ ]:
from sklearn.preprocessing import normalize
from sklearn.metrics import pairwise_distances
import numpy as np
import matplotlib.pyplot as plt

X_l2 = normalize(embedding, norm="l2")
D_cos = pairwise_distances(X_l2, metric="cosine")

# Prendo solo il triangolo superiore, escludendo la diagonale
n = D_cos.shape[0]
upper_idx = np.triu_indices(n, k=1)
pairwise_cosine_distances = D_cos[upper_idx]

# Istogramma
counts, bin_edges = np.histogram(
    pairwise_cosine_distances,
    bins=50,
    density=True
)

# 1. PICCO della distribuzione
peak_bin_index = np.argmax(counts)
peak_left = bin_edges[peak_bin_index]
peak_right = bin_edges[peak_bin_index + 1]
peak_center = (peak_left + peak_right) / 2

# 2. RANGE PRINCIPALE: 5°-95° percentile
main_range_left = np.quantile(pairwise_cosine_distances, 0.05)
main_range_right = np.quantile(pairwise_cosine_distances, 0.95)

# 3. CODA DESTRA: 95° percentile - massimo
right_tail_left = main_range_right
right_tail_right = pairwise_cosine_distances.max()

# Stampa metriche
print(f"Picco circa: {peak_center:.4f}")
print(f"Range principale circa: {main_range_left:.4f} - {main_range_right:.4f}")
print(f"Coda destra circa: {right_tail_left:.4f} - {right_tail_right:.4f}")

# Grafico
plt.figure(figsize=(7, 5))
plt.hist(
    pairwise_cosine_distances,
    bins=50,
    density=True,
    alpha=0.8
)

# Linee verticali per le tre metriche
plt.axvline(
    peak_center,
    linestyle="-",
    label=f"Picco ≈ {peak_center:.4f}"
)

plt.axvline(
    main_range_left,
    linestyle="--",
    label=f"5° percentile ≈ {main_range_left:.4f}"
)

plt.axvline(
    main_range_right,
    linestyle="--",
    label=f"95° percentile ≈ {main_range_right:.4f}"
)

plt.axvline(
    right_tail_right,
    linestyle=":",
    label=f"Max ≈ {right_tail_right:.4f}"
)

plt.xlabel("Cosine distance")
plt.ylabel("Density")
plt.title("Distribuzione delle distanze cosine tra glomeruli")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

Interpretazione della distribuzione delle distanze cosine

Questo grafico mostra la distribuzione delle distanze cosine tra tutte le coppie di glomeruli nello spazio degli embedding originali, dopo normalizzazione L2.

La distanza coseno è definita come:

cosine distance = 1 - cosine similarity

Valori bassi indicano che due glomeruli hanno embedding molto simili; valori più alti indicano che gli embedding sono più diversi.

Picco della distribuzione

Il picco rappresenta la distanza più frequente tra coppie di glomeruli.
Se il picco è molto basso, ad esempio vicino a 0.03–0.05, significa che gran parte dei campioni è molto simile in termini angolari. Questo può indicare uno spazio compatto, ma anche una possibile anisotropia degli embedding, cioè una tendenza dei vettori a puntare in una direzione comune.

Un picco più alto o una distribuzione più ampia suggeriscono invece che il backbone sta producendo rappresentazioni più differenziate tra i campioni.

Range principale

Il range principale è calcolato tra il 5° e il 95° percentile delle distanze.
Questo intervallo descrive dove si concentra la maggior parte delle coppie di glomeruli, ignorando gli estremi più rari.

Un range principale molto stretto indica che quasi tutte le coppie hanno distanze simili, quindi lo spazio potrebbe avere poca variabilità utile per il clustering.
Un range più ampio indica maggiore eterogeneità tra i campioni e può essere più favorevole alla separazione morfologica.

Coda destra

La coda destra va dal 95° percentile fino al valore massimo.
Questa parte della distribuzione rappresenta le coppie di glomeruli più distanti tra loro.

Una coda destra moderata è positiva, perché indica che alcuni glomeruli sono effettivamente diversi dal resto. Tuttavia, una coda molto lunga può anche essere dovuta a immagini problematiche, artefatti, maschere imprecise o campioni morfologicamente rari.

In [ ]:
from visualization.samples_plot import plot_nearest_neighbors_with_distances

plot_nearest_neighbors_with_distances(
    X=embedding,
    csv_path="/home/pasquale/PycharmProjects/Glomeruli-FP03-2026/data/glomeruli/embeddings/densenet_crops_embeddings.csv",
    query_indices=[2, 11, 135, 246, 391, 375],
    k=10,
    image_column="image_path",
    base_dir="/home/pasquale/PycharmProjects/Glomeruli-FP03-2026",
    metric="cosine",
    normalize_l2=True
)

In [ ]:
from metrics.nearest_neighbor import evaluate_embedding_backbone

metrics = evaluate_embedding_backbone(
    X=embedding,
    backbone_name="DenseNet169",
    ks=(5, 10, 20),
    hubness_k=10,
    metric="cosine",
    normalize_l2=True,
)

metrics

In [ ]:
from metrics.hopkins import compute_hopkins_dataframe

hopkins_df = compute_hopkins_dataframe(
    X=embedding,          # già PCA
    n_runs=100,
    sample_fraction=0.1,
    random_state=42,
    metric="euclidean",
)

hopkins_df